# LunarSite Stage 2: Terrain Segmentation (Kaggle v2)

**Runs in background -- close the tab, come back later.**

Checkpoints save to `/kaggle/working/` every epoch. Download results when done.

**Optimizations over v1 (mIoU 0.8425):**
- Focal+Dice loss with class weights (focus on large_rocks + sky)
- ResNet-50 encoder (more capacity)
- 480px input (safe for P100 16GB VRAM)
- Test-time augmentation (TTA) at inference

**Setup:** Settings > Accelerator > GPU P100

In [ ]:
# === INSTALL ALL DEPENDENCIES === #
!pip install -q segmentation-models-pytorch albumentations kagglehub psutil

In [ ]:
import gc
import json
import os
import time
from datetime import datetime
from pathlib import Path

import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
import cv2
import kagglehub
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import psutil
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')

# === OUTPUT DIRECTORY === #
OUT_DIR = '/kaggle/working/lunarsite_v2'
os.makedirs(f'{OUT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{OUT_DIR}/logs', exist_ok=True)
print(f'Output dir: {OUT_DIR}')

## Safety Utilities

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_metric, hist, tag):
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_metric': best_metric,
        'history': hist,
        'tag': tag,
        'timestamp': datetime.now().isoformat(),
    }
    torch.save(ckpt, f'{OUT_DIR}/checkpoints/{tag}_latest.pt')
    with open(f'{OUT_DIR}/logs/{tag}_history.json', 'w') as f:
        json.dump({'epoch': epoch, 'best_metric': best_metric, 'history': hist}, f, indent=2)


def save_best(model, epoch, best_metric, tag):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'best_metric': best_metric,
        'tag': tag,
    }, f'{OUT_DIR}/checkpoints/best_{tag}.pt')


def load_checkpoint(tag):
    path = f'{OUT_DIR}/checkpoints/{tag}_latest.pt'
    if os.path.exists(path):
        print(f'Found checkpoint: {path}')
        return torch.load(path, map_location=device, weights_only=False)
    return None


def write_status(tag, epoch, total_epochs, metric, phase='training'):
    status = {
        'tag': tag, 'phase': phase,
        'epoch': epoch, 'total_epochs': total_epochs,
        'best_metric': round(metric, 4),
        'last_update': datetime.now().isoformat(),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    }
    with open(f'{OUT_DIR}/logs/status.json', 'w') as f:
        json.dump(status, f, indent=2)


print('Safety utilities ready.')

## Dataset

In [ ]:
dataset_path = Path(kagglehub.dataset_download('romainpessia/artificial-lunar-rocky-landscape-dataset'))
image_dir = dataset_path / 'images' / 'render'
mask_dir = dataset_path / 'images' / 'clean'
real_moon_dir = dataset_path / 'real_moon_images'
print(f'Dataset: {dataset_path}')
print(f'Images: {len(list(image_dir.glob("render*.png")))}')
print(f'Real moon: {len(list(real_moon_dir.glob("*.png")))}')

In [ ]:
COLOR_TO_CLASS = {(0,0,0): 0, (255,0,0): 1, (0,255,0): 2, (0,0,255): 3}
CLASS_COLORS = np.array([[0,0,0],[255,165,0],[255,0,0],[135,206,235]], dtype=np.uint8)
CLASS_NAMES = ['background', 'small_rocks', 'large_rocks', 'sky']
NC = 4
SEED = 42

V1_MIOU = 0.8425
V1_PER_CLASS = {'background': 0.9754, 'small_rocks': 0.9748, 'large_rocks': 0.7092, 'sky': 0.7104}

## Lunar Augmentations + Dataset Class

In [ ]:
class LunarShadowRotation(ImageOnlyTransform):
    def __init__(self, angle_range=(-45, 45), p=0.5, **kwargs):
        super().__init__(p=p, **kwargs)
        self.angle_range = angle_range
    def apply(self, img, angle=0, **p):
        gray = np.mean(img.astype(np.float32), axis=-1)
        sh = gray < 25
        if sh.sum() < 100: return img
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        rot = cv2.warpAffine(sh.astype(np.uint8), M, (w, h), flags=cv2.INTER_NEAREST).astype(bool)
        r = img.copy()
        r[rot] = np.clip(r[rot] * 0.1, 0, 255).astype(np.uint8)
        r[sh & ~rot] = np.clip(r[sh & ~rot] * 3.0 + 30, 0, 200).astype(np.uint8)
        return r
    def get_params(self): return {'angle': np.random.uniform(*self.angle_range)}
    def get_transform_init_args_names(self): return ('angle_range',)


class ExtremeContrast(ImageOnlyTransform):
    def __init__(self, p=0.5, **kwargs):
        super().__init__(p=p, **kwargs)
    def apply(self, img, sf=0.05, bf=1.5, **p):
        g = np.mean(img.astype(np.float32), axis=-1)
        med = np.median(g)
        r = img.astype(np.float32)
        r[g < med * 0.5] *= sf
        r[g > med * 1.5] *= bf
        return np.clip(r, 0, 255).astype(np.uint8)
    def get_params(self):
        return {'sf': np.random.uniform(0.01, 0.15), 'bf': np.random.uniform(1.2, 2.5)}
    def get_transform_init_args_names(self): return ()


class HapkeBRDF(ImageOnlyTransform):
    def __init__(self, p=0.4, **kwargs):
        super().__init__(p=p, **kwargs)
    def apply(self, img, af=1.0, pf=1.0, **p):
        r = img.astype(np.float32) * af
        g = np.mean(r, axis=-1, keepdims=True)
        n = g / (g.max() + 1e-8)
        r *= (1 - n * (1 - n) * 4 * (1 - pf))
        return np.clip(r, 0, 255).astype(np.uint8)
    def get_params(self):
        return {'af': np.random.uniform(0.5, 1.5), 'pf': np.random.uniform(0.7, 1.3)}
    def get_transform_init_args_names(self): return ()


def get_transforms(sz, train):
    if train:
        return A.Compose([
            A.RandomCrop(sz, sz),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=(-0.3, 0.2), contrast_limit=(-0.2, 0.4), p=0.4),
            A.GaussNoise(p=0.2),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.3),
            LunarShadowRotation(p=0.3), ExtremeContrast(p=0.3), HapkeBRDF(p=0.3),
        ])
    return A.Compose([A.CenterCrop(sz, sz)])


class LunarDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        msk = cv2.cvtColor(cv2.imread(str(self.mask_paths[idx])), cv2.COLOR_BGR2RGB)
        h, w = msk.shape[:2]
        mask = np.zeros((h, w), dtype=np.int64)
        for color, cls in COLOR_TO_CLASS.items():
            mask[np.all(msk == color, axis=-1)] = cls
        if self.transform:
            t = self.transform(image=img, mask=mask)
            img, mask = t['image'], t['mask']
        return {
            'image': torch.from_numpy(img.transpose(2, 0, 1).astype(np.float32) / 255.0),
            'mask': torch.from_numpy(mask.astype(np.int64)),
        }

print('Augmentations + Dataset ready.')

In [ ]:
# Split dataset (deterministic -- same split as v1)
all_imgs = sorted(image_dir.glob('render*.png'))
all_masks = sorted(mask_dir.glob('clean*.png'))
n = len(all_imgs)
perm = np.random.RandomState(SEED).permutation(n).tolist()
nt, nv = int(n * 0.8), int(n * 0.1)
train_idx, val_idx, test_idx = perm[:nt], perm[nt:nt+nv], perm[nt+nv:]
print(f'Total: {n} | Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

## Compute Class Weights

In [ ]:
print('Computing class weights from training data...')
pixel_counts = np.zeros(NC, dtype=np.float64)
sample_idx = train_idx[:500]
for i in sample_idx:
    msk = cv2.cvtColor(cv2.imread(str(all_masks[i])), cv2.COLOR_BGR2RGB)
    for color, cls in COLOR_TO_CLASS.items():
        pixel_counts[cls] += np.all(msk == color, axis=-1).sum()

freq = pixel_counts / pixel_counts.sum()
weights = 1.0 / (freq + 1e-6)
weights = weights / weights.sum() * NC
class_weights = torch.FloatTensor(weights).to(device)

print('Class pixel frequencies:')
for i in range(NC):
    print(f'  {CLASS_NAMES[i]:15s}: {freq[i]:.4f} ({pixel_counts[i]/1e6:.1f}M px) -> weight {weights[i]:.3f}')

## Loss + Training Functions

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


class FocalDiceLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=2.0):
        super().__init__()
        self.focal = FocalLoss(alpha=class_weights, gamma=gamma)
        self.dice = smp.losses.DiceLoss(mode='multiclass')

    def forward(self, p, t):
        return 0.5 * self.focal(p, t) + 0.5 * self.dice(p, t)


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total = 0.0
    for batch in loader:
        imgs = batch['image'].to(device)
        masks = batch['mask'].to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * imgs.size(0)
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, nc):
    model.eval()
    total_loss = 0.0
    inter = torch.zeros(nc)
    uni = torch.zeros(nc)
    di = torch.zeros(nc)
    ds = torch.zeros(nc)
    for batch in loader:
        imgs = batch['image'].to(device)
        masks = batch['mask'].to(device)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
        total_loss += criterion(logits, masks).item() * imgs.size(0)
        preds = logits.argmax(1).cpu()
        mc = masks.cpu()
        for c in range(nc):
            pc, tc = (preds == c), (mc == c)
            inter[c] += (pc & tc).sum().float()
            uni[c] += (pc | tc).sum().float()
            di[c] += (pc.float() * tc.float()).sum()
            ds[c] += pc.float().sum() + tc.float().sum()
        del logits, preds, mc
    piou = [(inter[c] / uni[c]).item() if uni[c] > 0 else float('nan') for c in range(nc)]
    pdice = [(2 * di[c] / ds[c]).item() if ds[c] > 0 else float('nan') for c in range(nc)]
    vi = [x for x in piou if x == x]
    vd = [x for x in pdice if x == x]
    return {
        'loss': total_loss / len(loader.dataset),
        'per_class_iou': piou,
        'mean_iou': sum(vi) / len(vi) if vi else 0,
        'per_class_dice': pdice,
        'mean_dice': sum(vd) / len(vd) if vd else 0,
    }

print('Loss + training functions ready.')

In [ ]:
def run_training(model, train_ds, val_ds, epochs, lr, wd, tag, batch_size=6, num_workers=2):
    criterion = FocalDiceLoss(class_weights=class_weights, gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    scaler = torch.cuda.amp.GradScaler()

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)

    best = 0.0
    start_epoch = 1
    hist = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_dice': [], 'lr': []}

    ckpt = load_checkpoint(tag)
    if ckpt is not None:
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best = ckpt['best_metric']
        hist = ckpt['history']
        print(f'RESUMED {tag} from epoch {ckpt["epoch"]} (best mIoU: {best:.4f})')
        if start_epoch > epochs:
            print(f'{tag} already completed. Skipping.')
            return hist, best
    else:
        print(f'No checkpoint found for {tag}. Starting fresh.')

    print(f'\n{"="*90}')
    print(f'Training {tag} | epochs {start_epoch}-{epochs} | lr={lr} | batch={batch_size}')
    print(f'Loss: Focal(gamma=2) + Dice with class weights')
    print(f'Checkpoints: {OUT_DIR}/checkpoints/')
    print(f'{"-"*90}')

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()
        tl = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val = evaluate(model, val_loader, criterion, NC)
        scheduler.step()
        lr_now = optimizer.param_groups[0]['lr']
        dt = time.time() - t0

        hist['train_loss'].append(tl)
        hist['val_loss'].append(val['loss'])
        hist['val_miou'].append(val['mean_iou'])
        hist['val_dice'].append(val['mean_dice'])
        hist['lr'].append(lr_now)

        iou_str = ' | '.join(
            f"{CLASS_NAMES[i]}: {v:.3f}" if v == v else f"{CLASS_NAMES[i]}: N/A"
            for i, v in enumerate(val['per_class_iou'])
        )
        print(f'Epoch {epoch:3d}/{epochs} | train: {tl:.4f} | val: {val["loss"]:.4f} | '
              f'mIoU: {val["mean_iou"]:.4f} | Dice: {val["mean_dice"]:.4f} | '
              f'lr: {lr_now:.2e} | {dt:.0f}s')
        print(f'         {iou_str}', flush=True)

        if val['mean_iou'] > best:
            best = val['mean_iou']
            save_best(model, epoch, best, tag)
            print(f'         ** New best mIoU: {best:.4f} -- saved **')

        save_checkpoint(model, optimizer, scheduler, epoch, best, hist, tag)
        write_status(tag, epoch, epochs, best)

        gc.collect()
        torch.cuda.empty_cache()

    print(f'\n{tag} DONE. Best mIoU: {best:.4f}')
    write_status(tag, epochs, epochs, best, phase='complete')
    return hist, best

## Run 1: ResNet-50 Optimized

In [ ]:
INPUT_SIZE = 480

train_set = LunarDataset(
    [all_imgs[i] for i in train_idx], [all_masks[i] for i in train_idx],
    get_transforms(INPUT_SIZE, True))
val_set = LunarDataset(
    [all_imgs[i] for i in val_idx], [all_masks[i] for i in val_idx],
    get_transforms(INPUT_SIZE, False))

model_r50 = smp.Unet(
    encoder_name='resnet50', encoder_weights='imagenet',
    in_channels=3, classes=NC,
).to(device)
print(f'ResNet-50: {sum(p.numel() for p in model_r50.parameters() if p.requires_grad):,} params')

hist_r50, best_r50 = run_training(
    model_r50, train_set, val_set,
    epochs=60, lr=1e-4, wd=1e-4, tag='resnet50_v2',
    batch_size=6, num_workers=2,
)

## Run 2: DINOv2 Enhanced

In [ ]:
try:
    del model_r50
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()


class DINOv2UNet(nn.Module):
    def __init__(self, classes=4, dropout_p=0.1):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        self.embed_dim = self.backbone.embed_dim
        self.patch_size = 14
        n = len(self.backbone.blocks)
        self.feat_idx = [n // 4 - 1, n // 2 - 1, 3 * n // 4 - 1, n - 1]

        ch = [64, 128, 256, 512]
        self.projections = nn.ModuleList([
            nn.Sequential(nn.Conv2d(self.embed_dim, c, 1), nn.BatchNorm2d(c), nn.ReLU(True))
            for c in ch
        ])

        dec = [256, 128, 64, 32]
        self.decoders = nn.ModuleList()
        for i, dc in enumerate(dec):
            ic = ch[-(i + 1)] + (dec[i - 1] if i > 0 else 0)
            self.decoders.append(nn.Sequential(
                nn.Conv2d(ic, dc, 3, padding=1), nn.BatchNorm2d(dc), nn.ReLU(True),
                nn.Dropout2d(dropout_p),
                nn.Conv2d(dc, dc, 3, padding=1), nn.BatchNorm2d(dc), nn.ReLU(True),
            ))
        self.final = nn.Conv2d(dec[-1], classes, 1)

    def _extract(self, x):
        B, _, H, W = x.shape
        h, w = H // self.patch_size, W // self.patch_size
        feats = []
        hooks = []
        for idx in self.feat_idx:
            fl = []
            feats.append(fl)
            hooks.append(self.backbone.blocks[idx].register_forward_hook(
                lambda m, i, o, s=fl: s.append(o)
            ))
        self.backbone.forward_features(x)
        for hook in hooks:
            hook.remove()
        outs = []
        for i, (fl, proj) in enumerate(zip(feats, self.projections)):
            f = proj(fl[0][:, 1:, :].reshape(B, h, w, self.embed_dim).permute(0, 3, 1, 2))
            if i < 3:
                f = F.interpolate(f, scale_factor=1.0 / (2 ** (3 - i)),
                                  mode='bilinear', align_corners=False)
            outs.append(f)
        return outs

    def forward(self, x):
        _, _, H, W = x.shape
        enc = self._extract(x)
        d = self.decoders[0](enc[-1])
        for i in range(1, len(self.decoders)):
            d = F.interpolate(d, size=enc[-(i + 1)].shape[2:],
                              mode='bilinear', align_corners=False)
            d = self.decoders[i](torch.cat([d, enc[-(i + 1)]], dim=1))
        return self.final(F.interpolate(d, size=(H, W), mode='bilinear', align_corners=False))

In [ ]:
INPUT_SIZE_DINO = 476

train_set_dino = LunarDataset(
    [all_imgs[i] for i in train_idx], [all_masks[i] for i in train_idx],
    get_transforms(INPUT_SIZE_DINO, True))
val_set_dino = LunarDataset(
    [all_imgs[i] for i in val_idx], [all_masks[i] for i in val_idx],
    get_transforms(INPUT_SIZE_DINO, False))

model_dino = DINOv2UNet(classes=NC, dropout_p=0.1).to(device)
print(f'DINOv2-Base U-Net: {sum(p.numel() for p in model_dino.parameters() if p.requires_grad):,} params')

hist_dino, best_dino = run_training(
    model_dino, train_set_dino, val_set_dino,
    epochs=60, lr=5e-5, wd=1e-2, tag='dinov2_v2',
    batch_size=4, num_workers=2,
)

## Compare Results

In [ ]:
# Load history from disk if needed
for tname, hvar in [('resnet50_v2', 'r50'), ('dinov2_v2', 'dino')]:
    hpath = f'{OUT_DIR}/logs/{tname}_history.json'
    if os.path.exists(hpath):
        with open(hpath) as f:
            d = json.load(f)
        if hvar == 'r50' and (not hist_r50 or not hist_r50.get('val_miou')):
            hist_r50, best_r50 = d['history'], d['best_metric']
        elif hvar == 'dino' and (not hist_dino or not hist_dino.get('val_miou')):
            hist_dino, best_dino = d['history'], d['best_metric']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ep1 = range(1, len(hist_r50['val_miou']) + 1)
ep2 = range(1, len(hist_dino['val_miou']) + 1)

axes[0].plot(ep1, hist_r50['val_miou'], label='ResNet-50 v2', color='blue')
axes[0].plot(ep2, hist_dino['val_miou'], label='DINOv2 v2', color='red')
axes[0].axhline(y=V1_MIOU, color='green', linestyle='--', alpha=0.7, label=f'v1 baseline ({V1_MIOU})')
axes[0].set(xlabel='Epoch', ylabel='mIoU', title='Validation mIoU')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep1, hist_r50['val_dice'], label='ResNet-50 v2', color='blue')
axes[1].plot(ep2, hist_dino['val_dice'], label='DINOv2 v2', color='red')
axes[1].set(xlabel='Epoch', ylabel='Dice', title='Validation Dice')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep1, hist_r50['train_loss'], label='R50 train', color='blue', alpha=0.5)
axes[2].plot(ep1, hist_r50['val_loss'], label='R50 val', color='blue')
axes[2].plot(ep2, hist_dino['train_loss'], label='DINOv2 train', color='red', alpha=0.5)
axes[2].plot(ep2, hist_dino['val_loss'], label='DINOv2 val', color='red')
axes[2].set(xlabel='Epoch', ylabel='Loss', title='Training & Validation Loss')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'v2: ResNet-50 ({best_r50:.4f}) vs DINOv2 ({best_dino:.4f}) vs v1 ({V1_MIOU})', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/logs/comparison_v2.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Chart saved to {OUT_DIR}/logs/comparison_v2.png')

## Test-Time Augmentation (TTA)

In [ ]:
@torch.no_grad()
def predict_tta(model, img_t, nc=NC):
    model.eval()
    x = img_t.unsqueeze(0).to(device)
    augmented = [x, torch.flip(x, [3]), torch.flip(x, [2]), torch.flip(x, [2, 3])]
    probs_sum = None
    for i, aug_x in enumerate(augmented):
        with torch.cuda.amp.autocast():
            logits = model(aug_x)
        p = F.softmax(logits, dim=1)
        if i == 1: p = torch.flip(p, [3])
        elif i == 2: p = torch.flip(p, [2])
        elif i == 3: p = torch.flip(p, [2, 3])
        probs_sum = p if probs_sum is None else probs_sum + p
    return (probs_sum / len(augmented)).argmax(1).squeeze(0).cpu().numpy()


@torch.no_grad()
def evaluate_tta(model, loader, nc):
    model.eval()
    inter = torch.zeros(nc)
    uni = torch.zeros(nc)
    for batch in loader:
        imgs, masks = batch['image'], batch['mask']
        for j in range(imgs.size(0)):
            pred = predict_tta(model, imgs[j], nc)
            pred_t = torch.from_numpy(pred)
            gt = masks[j]
            for c in range(nc):
                pc, tc = (pred_t == c), (gt == c)
                inter[c] += (pc & tc).sum().float()
                uni[c] += (pc | tc).sum().float()
    piou = [(inter[c] / uni[c]).item() if uni[c] > 0 else float('nan') for c in range(nc)]
    vi = [x for x in piou if x == x]
    return {'per_class_iou': piou, 'mean_iou': sum(vi) / len(vi) if vi else 0}


@torch.no_grad()
def predict_one(model, img_t):
    model.eval()
    return model(img_t.unsqueeze(0).to(device)).argmax(1).squeeze(0).cpu().numpy()


def mc_predict(model, img_t, n_samples=20):
    model.eval()
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)):
            m.train()
    x = img_t.unsqueeze(0).to(device)
    probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            probs.append(F.softmax(model(x), dim=1).cpu())
    probs = torch.stack(probs).squeeze(1)
    mean_p = probs.mean(0)
    pred = mean_p.argmax(0).numpy()
    entropy = -(mean_p * torch.log(mean_p + 1e-10)).sum(0).numpy()
    return pred, entropy


print('TTA + prediction functions ready.')

## Test Evaluation (Standard + TTA)

In [ ]:
# Pick winner and load checkpoint
if best_dino >= best_r50:
    best_tag = 'dinov2_v2'
    best_model = DINOv2UNet(classes=NC, dropout_p=0.1).to(device)
    test_input_size = INPUT_SIZE_DINO
else:
    best_tag = 'resnet50_v2'
    best_model = smp.Unet(encoder_name='resnet50', encoder_weights=None,
                          in_channels=3, classes=NC).to(device)
    test_input_size = INPUT_SIZE

ckpt_path = f'{OUT_DIR}/checkpoints/best_{best_tag}.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
best_model.load_state_dict(ckpt['model_state_dict'])
print(f'Winner: {best_tag} (epoch {ckpt["epoch"]}, mIoU: {ckpt["best_metric"]:.4f})')

test_set_final = LunarDataset(
    [all_imgs[i] for i in test_idx], [all_masks[i] for i in test_idx],
    get_transforms(test_input_size, False))
test_loader = DataLoader(test_set_final, batch_size=6, shuffle=False,
                         num_workers=0, pin_memory=False)

# Standard eval
criterion = FocalDiceLoss(class_weights=class_weights, gamma=2.0)
test_std = evaluate(best_model, test_loader, criterion, NC)

print(f'\n--- Standard Test ---')
print(f'mIoU: {test_std["mean_iou"]:.4f}  |  Dice: {test_std["mean_dice"]:.4f}')
for i in range(NC):
    iv = f'{test_std["per_class_iou"][i]:.4f}' if test_std['per_class_iou'][i] == test_std['per_class_iou'][i] else 'N/A'
    print(f'  {CLASS_NAMES[i]:15s}: IoU={iv}  (v1: {V1_PER_CLASS[CLASS_NAMES[i]]:.4f})')

# TTA eval
print(f'\nRunning TTA evaluation...')
test_tta = evaluate_tta(best_model, test_loader, NC)

print(f'\n--- TTA Test ---')
print(f'mIoU: {test_tta["mean_iou"]:.4f}')
for i in range(NC):
    iv = f'{test_tta["per_class_iou"][i]:.4f}' if test_tta['per_class_iou'][i] == test_tta['per_class_iou'][i] else 'N/A'
    print(f'  {CLASS_NAMES[i]:15s}: IoU={iv}')

print(f'\n--- v1 vs v2 ---')
print(f'v1 mIoU:          {V1_MIOU:.4f}')
print(f'v2 Standard mIoU: {test_std["mean_iou"]:.4f}  ({test_std["mean_iou"]-V1_MIOU:+.4f})')
print(f'v2 TTA mIoU:      {test_tta["mean_iou"]:.4f}  ({test_tta["mean_iou"]-V1_MIOU:+.4f})')

# Save results
test_results = {
    'version': 'v2_kaggle',
    'best_model': best_tag,
    'r50_best_miou': round(best_r50, 4),
    'dinov2_best_miou': round(best_dino, 4),
    'best_epoch': ckpt['epoch'],
    'test_standard_miou': round(test_std['mean_iou'], 4),
    'test_standard_dice': round(test_std['mean_dice'], 4),
    'test_tta_miou': round(test_tta['mean_iou'], 4),
    'test_per_class_iou_standard': {CLASS_NAMES[i]: round(v, 4) if v == v else None
                                     for i, v in enumerate(test_std['per_class_iou'])},
    'test_per_class_iou_tta': {CLASS_NAMES[i]: round(v, 4) if v == v else None
                                for i, v in enumerate(test_tta['per_class_iou'])},
    'v1_comparison': {
        'v1_miou': V1_MIOU,
        'v1_per_class': V1_PER_CLASS,
        'improvement_standard': round(test_std['mean_iou'] - V1_MIOU, 4),
        'improvement_tta': round(test_tta['mean_iou'] - V1_MIOU, 4),
    },
}
with open(f'{OUT_DIR}/logs/test_results_v2.json', 'w') as f:
    json.dump(test_results, f, indent=2)
print(f'\nResults saved to {OUT_DIR}/logs/test_results_v2.json')

## Predictions + MC Dropout

In [ ]:
from matplotlib.patches import Patch

has_dropout = 'dinov2' in best_tag
ncols = 5 if has_dropout else 4
fig, axes = plt.subplots(4, ncols, figsize=(5 * ncols, 20))
titles = ['Input', 'Ground Truth', 'Prediction', 'TTA'] + (['Uncertainty'] if has_dropout else [])
for c, t in enumerate(titles):
    axes[0, c].set_title(t, fontsize=14, fontweight='bold')

for row in range(4):
    s = test_set_final[row * 50]
    img = s['image']
    gt = s['mask'].numpy()
    pred_std = predict_one(best_model, img)
    pred_tta_img = predict_tta(best_model, img)
    if has_dropout:
        _, unc = mc_predict(best_model, img)
    else:
        unc = None
    axes[row, 0].imshow(img.permute(1, 2, 0).numpy())
    axes[row, 1].imshow(CLASS_COLORS[gt])
    axes[row, 2].imshow(CLASS_COLORS[pred_std])
    axes[row, 3].imshow(CLASS_COLORS[pred_tta_img])
    if has_dropout:
        axes[row, 4].imshow(unc, cmap='hot')

for ax in axes.flat:
    ax.axis('off')
legend = [Patch(facecolor=c / 255, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=legend, loc='lower center', ncol=4, fontsize=12, bbox_to_anchor=(0.5, -0.02))
plt.suptitle(f'Predictions v2 ({best_tag})', fontsize=16)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/logs/predictions_v2.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Predictions saved to {OUT_DIR}/logs/predictions_v2.png')

## Sim-to-Real

In [ ]:
real_images = sorted(real_moon_dir.glob('*.png'))
fig, axes = plt.subplots(4, 3, figsize=(18, 20))
cols = ['Real Image', 'Standard', 'TTA']
for c, t in enumerate(cols):
    axes[0, c].set_title(t, fontsize=14, fontweight='bold')

for row in range(4):
    img = cv2.cvtColor(cv2.imread(str(real_images[row * 5])), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    cs = min(h, w, test_input_size)
    cropped = A.CenterCrop(cs, cs)(image=img)['image']
    if cs != test_input_size:
        cropped = cv2.resize(cropped, (test_input_size, test_input_size))
    img_t = torch.from_numpy(cropped.transpose(2, 0, 1).astype(np.float32) / 255.0)

    pred_std = predict_one(best_model, img_t)
    pred_tta_img = predict_tta(best_model, img_t)

    overlay_std = np.clip(0.6 * cropped / 255.0 + 0.4 * CLASS_COLORS[pred_std] / 255.0, 0, 1)
    overlay_tta = np.clip(0.6 * cropped / 255.0 + 0.4 * CLASS_COLORS[pred_tta_img] / 255.0, 0, 1)

    axes[row, 0].imshow(cropped)
    axes[row, 0].set_ylabel(real_images[row * 5].name, fontsize=9)
    axes[row, 1].imshow(overlay_std)
    axes[row, 2].imshow(overlay_tta)

for ax in axes.flat:
    ax.axis('off')
fig.legend(handles=legend, loc='lower center', ncol=4, fontsize=12, bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Sim-to-Real (v2)', fontsize=16)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/logs/sim_to_real_v2.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Sim-to-real saved to {OUT_DIR}/logs/sim_to_real_v2.png')

## Final Summary

All outputs are in `/kaggle/working/lunarsite_v2/`. Download from the **Output** tab.

In [ ]:
print('=' * 60)
print('  LUNARSITE STAGE 2 v2 -- FINAL RESULTS')
print('=' * 60)
print(f'  v1 ResNet-34 mIoU:     {V1_MIOU:.4f}')
print(f'  v2 ResNet-50 mIoU:     {best_r50:.4f}')
print(f'  v2 DINOv2 mIoU:        {best_dino:.4f}')
print(f'  Winner: {best_tag}')
print(f'  Test mIoU (standard):  {test_std["mean_iou"]:.4f}')
print(f'  Test mIoU (TTA):       {test_tta["mean_iou"]:.4f}')
print(f'  Improvement over v1:   {test_tta["mean_iou"]-V1_MIOU:+.4f}')
print('=' * 60)
print(f'\nPer-class IoU (v1 -> v2 TTA):')
for i in range(NC):
    v1 = V1_PER_CLASS[CLASS_NAMES[i]]
    v2 = test_tta['per_class_iou'][i]
    if v2 == v2:
        print(f'  {CLASS_NAMES[i]:15s}: {v1:.4f} -> {v2:.4f}  ({v2-v1:+.4f})')
print(f'\nDownload from Output tab:')
print(f'  lunarsite_v2/checkpoints/best_{best_tag}.pt')
print(f'  lunarsite_v2/logs/test_results_v2.json')